In [ ]:
# Célula 1 — Imports e configuração de caminhos
import pickle
import shutil
import tarfile
import urllib.request
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

import nltk
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

BASE_DIR    = Path().resolve().parent
DATA_DIR    = BASE_DIR / 'data'
MODELS_DIR  = BASE_DIR / 'models'
FIGURES_DIR = BASE_DIR / 'figures'

FIGURES_DIR.mkdir(exist_ok=True)
MODELS_DIR.mkdir(exist_ok=True)

ALGO_COLORS = {
    'Naive Bayes':         '#1C2B4A',
    'SVM':                 '#0891B2',
    'Random Forest':       '#2563EB',
    'Regressão Logística': '#64748B',
}

print(f'Projeto em: {BASE_DIR}')

In [ ]:
# Célula 2 — Reorganização dos arquivos existentes
CANDIDATES = [
    'X_train_tfidf.pkl', 'X_test_tfidf.pkl',
    'y_train.pkl', 'y_test.pkl', 'tfidf.pkl',
    'model_naive_bayes.pkl', 'model_svm.pkl',
    'model_random_forest.pkl', 'model_regressao.pkl',
    'results.csv',
]

exp0 = MODELS_DIR / 'exp0_enron_original'
existing = [f for f in CANDIDATES if (MODELS_DIR / f).exists()]

if existing:
    exp0.mkdir(exist_ok=True)
    print(f'Criada: {exp0}')
    for fname in existing:
        src = MODELS_DIR / fname
        dst = exp0 / fname
        if not dst.exists():
            shutil.move(str(src), str(dst))
            print(f'  Movido → exp0_enron_original/{fname}')
        else:
            print(f'  Já existe em exp0, pulando: {fname}')
else:
    print('Nenhum arquivo anterior encontrado na raiz de models/.')

NEW_DIRS = [
    'exp1_enron', 'exp1_spamassassin',
    'exp2_combined', 'exp3_crossdataset',
]
for d in NEW_DIRS:
    p = MODELS_DIR / d
    p.mkdir(exist_ok=True)
    print(f'Criada: {p}')

print('\nReorganização concluída.')

In [ ]:
# Célula 3 — Funções reutilizáveis de pré-processamento
STOP_WORDS = set(stopwords.words('english'))

def truncate_to_150_words(text):
    words = str(text).split()
    return ' '.join(words[:150])

def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t not in STOP_WORDS and len(t) > 1]
    return ' '.join(tokens)

def full_preprocess(text):
    return preprocess(truncate_to_150_words(text))

def build_models():
    return {
        'Naive Bayes':         MultinomialNB(),
        'SVM':                 LinearSVC(random_state=42, max_iter=2000, dual='auto'),
        'Random Forest':       RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
        'Regressão Logística': LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1),
    }

def train_evaluate(models, X_train, y_train, X_test, y_test):
    results = []
    for name, model in models.items():
        print(f'  Treinando: {name}...')
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        results.append({
            'Algoritmo': name,
            'Acurácia':  accuracy_score(y_test, y_pred),
            'Precisão':  precision_score(y_test, y_pred, pos_label=1, zero_division=0),
            'Recall':    recall_score(y_test, y_pred, pos_label=1, zero_division=0),
            'F1-score':  f1_score(y_test, y_pred, pos_label=1, zero_division=0),
        })
    return pd.DataFrame(results).set_index('Algoritmo')

def save_experiment(models, tfidf, X_train, X_test, y_train, y_test, df_results, dest_dir):
    dest_dir.mkdir(exist_ok=True)
    objs = {
        'tfidf': tfidf, 'X_train_tfidf': X_train, 'X_test_tfidf': X_test,
        'y_train': y_train, 'y_test': y_test,
    }
    for fname, obj in objs.items():
        with open(dest_dir / f'{fname}.pkl', 'wb') as f:
            pickle.dump(obj, f)
    for name, model in models.items():
        slug = name.lower().replace(' ', '_').replace('ã', 'a').replace('ç', 'c')
        with open(dest_dir / f'model_{slug}.pkl', 'wb') as f:
            pickle.dump(model, f)
    df_results.to_csv(dest_dir / 'results.csv')
    print(f'  Salvo em: {dest_dir}')

def plot_metrics(df_results, title, save_path):
    metrics = ['Acurácia', 'Precisão', 'Recall', 'F1-score']
    fig, axes = plt.subplots(1, 4, figsize=(16, 5))
    fig.suptitle(title, fontsize=13, fontweight='bold', y=1.02)
    for ax, metric in zip(axes, metrics):
        valores = df_results[metric].sort_values(ascending=True)
        colors  = [ALGO_COLORS[a] for a in valores.index]
        bars = ax.barh(valores.index, valores.values, color=colors, edgecolor='white')
        ax.set_xlim(max(0, valores.min() - 0.05), 1.0)
        ax.set_title(metric, fontweight='bold', fontsize=11)
        ax.set_xlabel('Score')
        for bar, val in zip(bars, valores.values):
            ax.text(val + 0.002, bar.get_y() + bar.get_height() / 2,
                    f'{val:.4f}', va='center', fontsize=9)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f'  Figura salva: {save_path}')

def run_cross_val(models_dict, X_train, y_train):
    print('  Validação cruzada 5-fold (F1-score):')
    cv_results = {}
    for name, model in models_dict.items():
        m = model.__class__(**model.get_params())
        scores = cross_val_score(m, X_train, y_train, cv=5, scoring='f1', n_jobs=-1)
        cv_results[name] = scores
        print(f'    {name:<25} média={scores.mean():.4f}  std={scores.std():.4f}')
    return cv_results

print('Funções carregadas.')

In [ ]:
# Célula 4 — Download e parsing do SpamAssassin
SA_CSV = DATA_DIR / 'spamassassin.csv'
SA_BASE_URL = 'https://spamassassin.apache.org/old/publiccorpus/'
SA_FILES = {
    '20030228_easy_ham.tar.bz2': 0,
    '20030228_spam.tar.bz2':     1,
}

def extract_email_body(raw_bytes):
    """Extrai o corpo do e-mail após o cabeçalho RFC 2822 (primeira linha em branco)."""
    try:
        text = raw_bytes.decode('latin-1')
    except Exception:
        text = raw_bytes.decode('utf-8', errors='ignore')
    parts = re.split(r'\n\n', text, maxsplit=1)
    return parts[1].strip() if len(parts) > 1 else ''

if SA_CSV.exists():
    print(f'SpamAssassin já processado: {SA_CSV}')
    df_sa_raw = pd.read_csv(SA_CSV)
    print(f'Total: {len(df_sa_raw)} e-mails')
    print(df_sa_raw['label'].value_counts().rename({1: 'spam', 0: 'ham'}))
else:
    records = []
    tmp_dir = DATA_DIR / '_sa_tmp'
    tmp_dir.mkdir(exist_ok=True)

    for filename, label in SA_FILES.items():
        tar_path = tmp_dir / filename
        if not tar_path.exists():
            url = SA_BASE_URL + filename
            print(f'Baixando {filename}...')
            urllib.request.urlretrieve(url, tar_path)
            print(f'  Download concluído: {tar_path}')
        else:
            print(f'  Arquivo já existe: {tar_path}')

        print(f'  Extraindo {filename}...')
        with tarfile.open(tar_path, 'r:bz2') as tar:
            for member in tar.getmembers():
                if member.isfile() and not member.name.endswith('cmds'):
                    f = tar.extractfile(member)
                    if f is not None:
                        body = extract_email_body(f.read())
                        if body.strip():
                            records.append({'Message': body, 'label': label})
        print(f'  E-mails extraídos até agora: {len(records)}')

    df_sa_raw = pd.DataFrame(records)
    df_sa_raw.to_csv(SA_CSV, index=False)
    print(f'\nSalvo em: {SA_CSV}')
    print(f'Total: {len(df_sa_raw)} e-mails')
    print(df_sa_raw['label'].value_counts().rename({1: 'spam', 0: 'ham'}))

    shutil.rmtree(tmp_dir, ignore_errors=True)

In [ ]:
# Célula 5 — Análise exploratória rápida dos dois datasets

# --- Enron ---
df_enron_raw = pd.read_csv(DATA_DIR / 'enron_spam_data.csv')
df_enron_raw = df_enron_raw.dropna(subset=['Message'])
df_enron_raw['label'] = df_enron_raw['Spam/Ham'].map({'spam': 1, 'ham': 0})
df_enron_raw = df_enron_raw[df_enron_raw['label'].notna()]

print('=== Enron ===')
print(f'Total de e-mails: {len(df_enron_raw)}')
print(df_enron_raw['label'].value_counts().rename({1: 'spam', 0: 'ham'}))
print(f'Proporção spam: {df_enron_raw["label"].mean():.2%}')

# --- SpamAssassin ---
df_sa = pd.read_csv(SA_CSV)
df_sa = df_sa.dropna(subset=['Message'])

print('\n=== SpamAssassin ===')
print(f'Total de e-mails: {len(df_sa)}')
print(df_sa['label'].value_counts().rename({1: 'spam', 0: 'ham'}))
print(f'Proporção spam: {df_sa["label"].mean():.2%}')

# --- Comprimento médio dos corpos (palavras) ---
df_enron_raw['n_words'] = df_enron_raw['Message'].apply(lambda x: len(str(x).split()))
df_sa['n_words']        = df_sa['Message'].apply(lambda x: len(str(x).split()))

print('\n=== Comprimento dos e-mails (palavras, corpo completo) ===')
print(f'Enron   — mediana: {df_enron_raw["n_words"].median():.0f}  média: {df_enron_raw["n_words"].mean():.0f}')
print(f'SpamAssassin — mediana: {df_sa["n_words"].median():.0f}  média: {df_sa["n_words"].mean():.0f}')

In [ ]:
# Célula 6 — Experimento 1: Enron — Pré-processamento e split
print('=== EXPERIMENTO 1 — Enron (isolado) ===')

df_enron = df_enron_raw[['Message', 'label']].copy()
df_enron['text'] = df_enron['Message'].apply(full_preprocess)
df_enron = df_enron[df_enron['text'].str.strip() != '']

X_e = df_enron['text']
y_e = df_enron['label'].astype(int)

X_e_train, X_e_test, y_e_train, y_e_test = train_test_split(
    X_e, y_e, test_size=0.2, random_state=42, stratify=y_e
)

tfidf_e = TfidfVectorizer(max_features=10000, sublinear_tf=True, min_df=2, ngram_range=(1, 2))
X_e_train_tfidf = tfidf_e.fit_transform(X_e_train)   # ajuste APENAS no treino
X_e_test_tfidf  = tfidf_e.transform(X_e_test)

print(f'Treino : {X_e_train_tfidf.shape}')
print(f'Teste  : {X_e_test_tfidf.shape}')
print(f'Distribuição treino — spam: {y_e_train.mean():.2%}')

In [ ]:
# Célula 7 — Experimento 1: Enron — Treinamento e avaliação
models_e1 = build_models()
df_e1 = train_evaluate(models_e1, X_e_train_tfidf, y_e_train, X_e_test_tfidf, y_e_test)
df_e1 = df_e1.sort_values('F1-score', ascending=False)

print('\nResultados — Enron:')
print(df_e1.round(4).to_string())

save_experiment(
    models_e1, tfidf_e,
    X_e_train_tfidf, X_e_test_tfidf,
    y_e_train, y_e_test,
    df_e1, MODELS_DIR / 'exp1_enron'
)

In [ ]:
# Célula 8 — Experimento 1: Enron — Visualização e validação cruzada
plot_metrics(
    df_e1,
    'Experimento 1 — Desempenho por Algoritmo (Enron, recorte 150 palavras)',
    FIGURES_DIR / 'exp1_enron.png'
)

cv_e1 = run_cross_val(models_e1, X_e_train_tfidf, y_e_train)

In [ ]:
# Célula 9 — Experimento 1: SpamAssassin — Pré-processamento e split
print('=== EXPERIMENTO 1 — SpamAssassin (isolado) ===')

df_sa_proc = df_sa[['Message', 'label']].copy()
df_sa_proc['text'] = df_sa_proc['Message'].apply(full_preprocess)
df_sa_proc = df_sa_proc[df_sa_proc['text'].str.strip() != '']

X_s = df_sa_proc['text']
y_s = df_sa_proc['label'].astype(int)

X_s_train, X_s_test, y_s_train, y_s_test = train_test_split(
    X_s, y_s, test_size=0.2, random_state=42, stratify=y_s
)

tfidf_s = TfidfVectorizer(max_features=10000, sublinear_tf=True, min_df=2, ngram_range=(1, 2))
X_s_train_tfidf = tfidf_s.fit_transform(X_s_train)   # ajuste APENAS no treino
X_s_test_tfidf  = tfidf_s.transform(X_s_test)

print(f'Treino : {X_s_train_tfidf.shape}')
print(f'Teste  : {X_s_test_tfidf.shape}')
print(f'Distribuição treino — spam: {y_s_train.mean():.2%}')

In [ ]:
# Célula 10 — Experimento 1: SpamAssassin — Treinamento e avaliação
models_s1 = build_models()
df_s1 = train_evaluate(models_s1, X_s_train_tfidf, y_s_train, X_s_test_tfidf, y_s_test)
df_s1 = df_s1.sort_values('F1-score', ascending=False)

print('\nResultados — SpamAssassin:')
print(df_s1.round(4).to_string())

save_experiment(
    models_s1, tfidf_s,
    X_s_train_tfidf, X_s_test_tfidf,
    y_s_train, y_s_test,
    df_s1, MODELS_DIR / 'exp1_spamassassin'
)

In [ ]:
# Célula 11 — Experimento 1: SpamAssassin — Visualização e validação cruzada
plot_metrics(
    df_s1,
    'Experimento 1 — Desempenho por Algoritmo (SpamAssassin, recorte 150 palavras)',
    FIGURES_DIR / 'exp1_spamassassin.png'
)

cv_s1 = run_cross_val(models_s1, X_s_train_tfidf, y_s_train)

In [ ]:
# Célula 12 — Experimento 2: Combinado — Concatenação e split
print('=== EXPERIMENTO 2 — Dataset Combinado ===')

df_enron_tmp = df_enron[['text', 'label']].copy()
df_sa_tmp    = df_sa_proc[['text', 'label']].copy()
df_combined  = pd.concat([df_enron_tmp, df_sa_tmp], ignore_index=True)

print(f'Total combinado: {len(df_combined)} e-mails')
print(df_combined['label'].value_counts().rename({1: 'spam', 0: 'ham'}))

X_c = df_combined['text']
y_c = df_combined['label'].astype(int)

X_c_train, X_c_test, y_c_train, y_c_test = train_test_split(
    X_c, y_c, test_size=0.2, random_state=42, stratify=y_c
)

tfidf_c = TfidfVectorizer(max_features=10000, sublinear_tf=True, min_df=2, ngram_range=(1, 2))
X_c_train_tfidf = tfidf_c.fit_transform(X_c_train)   # ajuste APENAS no treino
X_c_test_tfidf  = tfidf_c.transform(X_c_test)

print(f'\nTreino : {X_c_train_tfidf.shape}')
print(f'Teste  : {X_c_test_tfidf.shape}')

In [ ]:
# Célula 13 — Experimento 2: Combinado — Treinamento e avaliação
models_c2 = build_models()
df_c2 = train_evaluate(models_c2, X_c_train_tfidf, y_c_train, X_c_test_tfidf, y_c_test)
df_c2 = df_c2.sort_values('F1-score', ascending=False)

print('\nResultados — Combinado:')
print(df_c2.round(4).to_string())

save_experiment(
    models_c2, tfidf_c,
    X_c_train_tfidf, X_c_test_tfidf,
    y_c_train, y_c_test,
    df_c2, MODELS_DIR / 'exp2_combined'
)

In [ ]:
# Célula 14 — Experimento 2: Combinado — Visualização
plot_metrics(
    df_c2,
    'Experimento 2 — Desempenho por Algoritmo (Corpus Combinado, recorte 150 palavras)',
    FIGURES_DIR / 'exp2_combined.png'
)

In [ ]:
# Célula 15 — Experimento 3: Cross-dataset — Preparação
# O TF-IDF é ajustado no treino do Enron e aplicado NO SpamAssassin sem reajuste.
# Isso mede a capacidade de generalização para um corpus fora da distribuição de treino.
print('=== EXPERIMENTO 3 — Cross-dataset (Enron → SpamAssassin) ===')

# Reutiliza o TF-IDF e split do Experimento 1 Enron (tfidf_e, X_e_train_tfidf, y_e_train)
# Aplica transform (nunca fit) no SpamAssassin completo como conjunto de teste externo
X_sa_full_text = df_sa_proc['text']
y_sa_full      = df_sa_proc['label'].astype(int)

# TF-IDF do Enron aplicado ao SpamAssassin — vocabulário do Enron, sem reajuste
X_sa_ext_tfidf = tfidf_e.transform(X_sa_full_text)

print(f'Treino (Enron 80%) : {X_e_train_tfidf.shape}')
print(f'Teste externo (SpamAssassin completo): {X_sa_ext_tfidf.shape}')
print(f'Distribuição teste externo — spam: {y_sa_full.mean():.2%}')

In [ ]:
# Célula 16 — Experimento 3: Cross-dataset — Treinamento e avaliação
models_x3 = build_models()
df_x3 = train_evaluate(models_x3, X_e_train_tfidf, y_e_train, X_sa_ext_tfidf, y_sa_full)
df_x3 = df_x3.sort_values('F1-score', ascending=False)

print('\nResultados — Cross-dataset (Enron treino → SpamAssassin teste):')
print(df_x3.round(4).to_string())

dest_x3 = MODELS_DIR / 'exp3_crossdataset'
dest_x3.mkdir(exist_ok=True)
for name, model in models_x3.items():
    slug = name.lower().replace(' ', '_').replace('ã', 'a').replace('ç', 'c')
    with open(dest_x3 / f'model_{slug}.pkl', 'wb') as f:
        pickle.dump(model, f)
df_x3.to_csv(dest_x3 / 'results.csv')
print(f'  Salvo em: {dest_x3}')

In [ ]:
# Célula 17 — Experimento 3: Cross-dataset — Visualização
plot_metrics(
    df_x3,
    'Experimento 3 — Generalização Cross-dataset (treino Enron, teste SpamAssassin)',
    FIGURES_DIR / 'exp3_crossdataset.png'
)

In [ ]:
# Célula 18 — Consolidação final e figura resumo
EXP_LABELS = {
    'Enron':         df_e1,
    'SpamAssassin':  df_s1,
    'Combinado':     df_c2,
    'Cross-dataset': df_x3,
}

all_records = []
for exp_name, df_exp in EXP_LABELS.items():
    for algo, row in df_exp.iterrows():
        all_records.append({
            'Experimento': exp_name,
            'Algoritmo':   algo,
            'Acurácia':    row['Acurácia'],
            'Precisão':    row['Precisão'],
            'Recall':      row['Recall'],
            'F1-score':    row['F1-score'],
        })

df_all = pd.DataFrame(all_records)
df_all.to_csv(MODELS_DIR / 'all_results.csv', index=False)
print('Salvo: models/all_results.csv')
print(df_all.to_string(index=False))

# --- Figura resumo: F1-score por algoritmo em cada experimento ---
algos = list(ALGO_COLORS.keys())
exps  = list(EXP_LABELS.keys())

fig, axes = plt.subplots(1, 4, figsize=(18, 5), sharey=False)
fig.suptitle(
    'Comparação Final — F1-score por Experimento e Algoritmo',
    fontsize=13, fontweight='bold', y=1.02
)

for ax, (exp_name, df_exp) in zip(axes, EXP_LABELS.items()):
    for algo in algos:
        if algo in df_exp.index:
            val = df_exp.loc[algo, 'F1-score']
            ax.bar(algo, val, color=ALGO_COLORS[algo], edgecolor='white')
            ax.text(algo, val + 0.005, f'{val:.3f}', ha='center', va='bottom', fontsize=8)
    ax.set_title(exp_name, fontweight='bold')
    ax.set_ylabel('F1-score')
    ax.set_ylim(0, 1.05)
    ax.set_xticklabels(
        [a.replace(' ', '\n') for a in algos],
        rotation=0, fontsize=8
    )
    ax.set_xticks(range(len(algos)))
    ax.set_xticklabels([a.replace(' ', '\n') for a in algos], fontsize=8)

handles = [plt.Rectangle((0, 0), 1, 1, color=ALGO_COLORS[a]) for a in algos]
fig.legend(handles, algos, loc='lower center', ncol=4, bbox_to_anchor=(0.5, -0.08), fontsize=10)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'final_comparison.png', dpi=300, bbox_inches='tight')
plt.show()
print('Figura salva: figures/final_comparison.png')

In [ ]:
# Célula 19 — Resumo textual dos achados
pivot = df_all.pivot(index='Algoritmo', columns='Experimento', values='F1-score')

# Consistência = menor desvio padrão entre os quatro experimentos
pivot['std'] = pivot[exps].std(axis=1)
mais_consistente = pivot['std'].idxmin()

# Queda no cross-dataset vs melhor experimento individual
pivot['queda_cross'] = pivot[['Enron', 'SpamAssassin', 'Combinado']].max(axis=1) - pivot['Cross-dataset']
maior_queda = pivot['queda_cross'].idxmax()

# Comparação NB vs SVM/RF
nb_mean  = pivot.loc['Naive Bayes', exps].mean()
svm_mean = pivot.loc['SVM', exps].mean()
rf_mean  = pivot.loc['Random Forest', exps].mean()
hipotese_confirmada = (nb_mean < svm_mean) and (nb_mean < rf_mean)

print('=' * 65)
print('RESUMO FINAL DOS ACHADOS')
print('=' * 65)
print(f'\nF1-score médio por algoritmo (todos os experimentos):')
for algo in algos:
    print(f'  {algo:<25} {pivot.loc[algo, exps].mean():.4f}')

print(f'\n• Algoritmo mais consistente entre experimentos: {mais_consistente}')
print(f'  (menor desvio padrão de F1-score: {pivot.loc[mais_consistente, "std"]:.4f})')

print(f'\n• Maior queda no cross-dataset: {maior_queda}')
print(f'  (queda de {pivot.loc[maior_queda, "queda_cross"]:.4f} pontos de F1 vs melhor individual)')

print(f'\n• Hipótese do trabalho (NB inferior, SVM/RF superiores): ', end='')
print('CONFIRMADA' if hipotese_confirmada else 'NÃO CONFIRMADA')
print(f'  NB médio = {nb_mean:.4f} | SVM médio = {svm_mean:.4f} | RF médio = {rf_mean:.4f}')

print('\n' + '=' * 65)